In [2]:
import pandas as pd
import numpy as np
# --- Tabela 1: autorzy ---
autorzy = pd.DataFrame({
 'autor_id': [1, 2, 3, 4, 5, 6, 7, 8],
 'imie': ['Olga', 'Stanisław', 'Andrzej', 'Wisława', 'Ryszard',
 'Dorota', 'Szczepan', 'Jacek'],
 'nazwisko': ['Tokarczuk', 'Lem', 'Sapkowski', 'Szymborska', 'Kapuściński',
 'Masłowska', 'Twardoch', 'Dehnel'],
 'kraj': ['Polska', 'Polska', 'Polska', 'Polska', 'Polska',
 'Polska', 'Polska', 'Polska'],
 'nagrody': ['Nobel', 'SFF', 'SFF', 'Nobel', 'Reporter',
 'Polityka', 'NIKE', 'NIKE']
})
# --- Tabela 2: książki ---
ksiazki = pd.DataFrame({
 'ksiazka_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
 'autor_id': [1, 1, 2, 2, 3, 3, 4, 5, 6, 7, 7, 8],
 'tytul': ['Księgi Jakubowe', 'Bieguni', 'Solaris', 'Cyberiada',
 'Wiedźmin', 'Narrenturm', 'Wiersze wybrane', 'Podróże z Herodotem',
 'Wojna polsko-ruska', 'Morfina', 'Król', 'Lala'],
 'kategoria': ['historyczna', 'obyczajowa', 'sci-fi', 'sci-fi',
 'fantasy', 'fantasy', 'poezja', 'reportaz',
 'obyczajowa', 'historyczna', 'historyczna', 'obyczajowa'],
 'cena': [79.90, 49.00, 39.00, 42.00, 45.00, 55.00, 35.00,
 52.00, 38.00, 48.00, 59.00, 44.00],
 'strony': [912, 376, 198, 295, 320, 512, 180, 263, 153, 618, 688, 432]
})
# --- Tabela 3: zamówienia ---
np.random.seed(2026)
n = 80
zamowienia = pd.DataFrame({
 'zam_id': range(5001, 5001 + n),
 'ksiazka_id': np.random.choice(ksiazki['ksiazka_id'], size=n),
 'ilosc': np.random.randint(1, 6, size=n),
 'data': pd.to_datetime('2026-01-01') +
pd.to_timedelta(np.random.randint(0, 120, n), unit='D'),
 'kanal': np.random.choice(['web', 'aplikacja', 'telefon'], size=n, p=
[0.6, 0.3, 0.1]),
 'miasto': np.random.choice(['Warszawa', 'Kraków', 'Wrocław', 'Gdańsk',
'Poznań'], size=n)
})
print(f"Autorzy: {autorzy.shape}")
print(f"Ksiazki: {ksiazki.shape}")
print(f"Zamowienia: {zamowienia.shape}")


Autorzy: (8, 5)
Ksiazki: (12, 6)
Zamowienia: (80, 6)


## Ćwiczenie 1: merge() 

Krok 1 — prosty merge (inner)

#### Zadanie 1a

In [3]:
# Połącz ksiazki + autorzy po kluczu autor_id
ksiazki_z_autorami = ksiazki.merge(autorzy, on='autor_id')
print(f"Shape: {ksiazki_z_autorami.shape}")
ksiazki_z_autorami.head()

Shape: (12, 10)


,ksiazka_id,autor_id,tytul,kategoria,cena,strony,imie,nazwisko,kraj,nagrody
0,101,1,Księgi Jakubowe,historyczna,79.9,912,Olga,Tokarczuk,Polska,Nobel
1,102,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel
2,103,2,Solaris,sci-fi,39.0,198,Stanisław,Lem,Polska,SFF
3,104,2,Cyberiada,sci-fi,42.0,295,Stanisław,Lem,Polska,SFF
4,105,3,Wiedźmin,fantasy,45.0,320,Andrzej,Sapkowski,Polska,SFF


Ile wierszy ma wynik? 12
Ile kolumn? 10
Jakie nowe kolumny pojawiły się (z tabeli autorzy)? imie, nazwisko, kraj, nagrody

Krok 2 — cztery typy how

In [4]:
ksiazki_test = pd.concat([
 ksiazki,
 pd.DataFrame({'ksiazka_id':[999], 'autor_id':[999],
 'tytul':['Ksiazka bez autora'], 'kategoria':['test'],
 'cena':[30.0], 'strony':[100]})
], ignore_index=True)

Zadanie 1b: Porównaj cztery typy złączenia:

In [5]:
inner = ksiazki_test.merge(autorzy, on='autor_id', how='inner')
left = ksiazki_test.merge(autorzy, on='autor_id', how='left')
right = ksiazki_test.merge(autorzy, on='autor_id', how='right')
outer = ksiazki_test.merge(autorzy, on='autor_id', how='outer')
print(f"inner: {inner.shape[0]} wierszy")
print(f"left: {left.shape[0]} wierszy")
print(f"right: {right.shape[0]} wierszy")
print(f"outer: {outer.shape[0]} wierszy")

inner: 12 wierszy
left: 13 wierszy
right: 12 wierszy
outer: 13 wierszy


Krok 3 — diagnostyka z indicator=True

In [6]:
audyt = ksiazki_test.merge(autorzy, on='autor_id', how='outer', indicator=True)
audyt['_merge'].value_counts()

_merge
both          12
left_only      1
right_only     0
Name: count, dtype: int64

inner dosasowywuje tabele autor z tabelą książki patrząc na autor_id. Książka z autor_id=999 nie ma przypisanego autora w tabeli autor, więc zostaje pominęta. left natomiast zachowuje tą książke z NaN w kolumnach z tabeli autorzy.

## Ćwiczenie 2: Merge łańcuchowy + kolumny wyliczne

Krok 1 — łańcuch 3 tabel

In [7]:
# Połącz: zamowienia + ksiazki + autorzy
pelne = (
 zamowienia
 .merge(ksiazki, on='ksiazka_id')
 .merge(autorzy, on='autor_id')
)
print(f"Pelna tabela: {pelne.shape}")
pelne.head()

Pelna tabela: (80, 15)


,zam_id,ksiazka_id,ilosc,data,kanal,miasto,autor_id,tytul,kategoria,cena,strony,imie,nazwisko,kraj,nagrody
0,5001,102,5,2026-04-05,web,Poznań,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel
1,5017,102,5,2026-04-20,web,Wrocław,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel
2,5024,102,1,2026-01-28,telefon,Wrocław,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel
3,5038,102,2,2026-04-14,web,Kraków,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel
4,5042,102,1,2026-04-11,telefon,Warszawa,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel


Krok 2 — kolumny wyliczane

Zadanie 2a: Dodaj cztery nowe kolumny:

In [8]:
# 1. Wartość zamówienia
pelne['wartosc'] = pelne['ilosc'] * pelne['cena']
print(pelne['wartosc'])
# 2. Miesiąc zamówienia
pelne['miesiac'] = pelne['data'].dt.month
print(pelne['miesiac'])
# 3. Pełne imię i nazwisko autora
pelne['autor_pelne'] = pelne['imie'] + ' ' + pelne['nazwisko']
print(pelne['autor_pelne'])
# 4. Kategoria cenowa książki
pelne['kategoria_cenowa'] = np.where(pelne['cena'] >= 50, 'droga', 'tania')
print(pelne['kategoria_cenowa'])

0     245.0
1     245.0
2      49.0
3      98.0
4      49.0
      ...  
75    208.0
76    260.0
77    156.0
78    260.0
79    260.0
Name: wartosc, Length: 80, dtype: float64
0     4
1     4
2     1
3     4
4     4
     ..
75    3
76    3
77    1
78    1
79    3
Name: miesiac, Length: 80, dtype: int32
0          Olga Tokarczuk
1          Olga Tokarczuk
2          Olga Tokarczuk
3          Olga Tokarczuk
4          Olga Tokarczuk
             ...         
75    Ryszard Kapuściński
76    Ryszard Kapuściński
77    Ryszard Kapuściński
78    Ryszard Kapuściński
79    Ryszard Kapuściński
Name: autor_pelne, Length: 80, dtype: object
0     tania
1     tania
2     tania
3     tania
4     tania
      ...  
75    droga
76    droga
77    droga
78    droga
79    droga
Name: kategoria_cenowa, Length: 80, dtype: object


In [10]:
print(f"Pelna tabela: {pelne.shape}")
pelne.head()

Pelna tabela: (80, 19)


,zam_id,ksiazka_id,ilosc,data,kanal,miasto,autor_id,tytul,kategoria,cena,strony,imie,nazwisko,kraj,nagrody,wartosc,miesiac,autor_pelne,kategoria_cenowa
0,5001,102,5,2026-04-05,web,Poznań,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel,245.0,4,Olga Tokarczuk,tania
1,5017,102,5,2026-04-20,web,Wrocław,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel,245.0,4,Olga Tokarczuk,tania
2,5024,102,1,2026-01-28,telefon,Wrocław,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel,49.0,1,Olga Tokarczuk,tania
3,5038,102,2,2026-04-14,web,Kraków,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel,98.0,4,Olga Tokarczuk,tania
4,5042,102,1,2026-04-11,telefon,Warszawa,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel,49.0,4,Olga Tokarczuk,tania


Krok 3 — pierwsze wnioski

Zadanie 2b: Odpowiedz na pytania:

In [14]:
# 1. Ile jest wszystkich zamówień?
print(zamowienia.shape[0])
# 2. Jaki jest łączny przychód?
print(pelne['wartosc'].sum())
# 3. Jaka jest średnia wartość zamówienia?
print(pelne['wartosc'].mean())
# 4. Ile książek (unikalnych tytułów) się sprzedało? (nunique)
print(ksiazki['tytul'].nunique())

80
11963.8
149.54749999999999
12


## Ćwiczenie 3: groupby + .agg() — raporty per kategoria

Krok 1 — podstawowy groupby

Zadanie 3a: Dla każdej kategoria policz łączny przychód (suma wartosc) i posortuj malejąco:

In [15]:
pelne.groupby('kategoria')['wartosc'].sum().sort_values(ascending=False)

kategoria
historyczna    4136.8
sci-fi         2334.0
obyczajowa     1999.0
fantasy        1930.0
reportaz       1144.0
poezja          420.0
Name: wartosc, dtype: float64

Krok 2 — wiele funkcji naraz

Zadanie 3b: Dla każdej kategoria policz: liczbę zamówień, średnią wartość, łączną wartość:

In [23]:
pelne.groupby('kategoria')['wartosc'].agg(['count', 'mean', 'sum']).round(2)

,count,mean,sum
kategoria,,,
fantasy,14,137.86,1930.0
historyczna,22,188.04,4136.8
obyczajowa,17,117.59,1999.0
poezja,7,60.00,420.0
reportaz,5,228.80,1144.0
sci-fi,15,155.60,2334.0


Krok 3 — named aggregation (styl nowoczesny)

Zadanie 3c: Zbuduj raport per autor używając named aggregation

In [26]:
pelne.groupby('autor_pelne').agg(
    liczba_zamówien=('zam_id', 'count'),
    laczna_sprzedaz=('wartosc', 'sum'),
    srednia_ilosc=('ilosc', 'mean')
).round(2).sort_values('laczna_sprzedaz', ascending=False)

,liczba_zamówien,laczna_sprzedaz,srednia_ilosc
autor_pelne,,,
Olga Tokarczuk,18,3389.8,2.72
Stanisław Lem,15,2334.0,3.80
Andrzej Sapkowski,14,1930.0,2.86
Szczepan Twardoch,11,1580.0,2.91
Ryszard Kapuściński,5,1144.0,4.40
Jacek Dehnel,5,748.0,3.40
Wisława Szymborska,7,420.0,1.71
Dorota Masłowska,5,418.0,2.20


Krok 4 — groupby po wielu kolumnach

Zadanie 3d: Policz łączną wartość zamówień per kategoria × kanal:

In [27]:
pelne.groupby(['kategoria', 'kanal'])['wartosc'].sum()

kategoria    kanal    
fantasy      aplikacja     475.0
             telefon       225.0
             web          1230.0
historyczna  aplikacja    1988.5
             telefon       527.4
             web          1620.9
obyczajowa   aplikacja     586.0
             telefon       136.0
             web          1277.0
poezja       aplikacja      70.0
             telefon        70.0
             web           280.0
reportaz     aplikacja     416.0
             telefon       260.0
             web           468.0
sci-fi       aplikacja     534.0
             telefon       279.0
             web          1521.0
Name: wartosc, dtype: float64

Krok 5 — udział procentowy (transform)

Zadanie 3e: Dodaj do pelne kolumnę udzial_w_kategorii_pct:

In [ ]:
pelne['wartosc_kat'] = pelne.groupby('kategoria')['wartosc'].transform('sum')
pelne['udzial_w_kategorii_pct'] = (pelne['wartosc'] / pelne['wartosc_kat'] *100).round(2)

In [29]:
print(f"Pelna tabela: {pelne.shape}")
pelne.head()

Pelna tabela: (80, 21)


,zam_id,ksiazka_id,ilosc,data,kanal,miasto,autor_id,tytul,kategoria,cena,...,imie,nazwisko,kraj,nagrody,wartosc,miesiac,autor_pelne,kategoria_cenowa,wartosc_kat,udzial_w_kategorii_pct
0,5001,102,5,2026-04-05,web,Poznań,1,Bieguni,obyczajowa,49.0,...,Olga,Tokarczuk,Polska,Nobel,245.0,4,Olga Tokarczuk,tania,1999.0,12.26
1,5017,102,5,2026-04-20,web,Wrocław,1,Bieguni,obyczajowa,49.0,...,Olga,Tokarczuk,Polska,Nobel,245.0,4,Olga Tokarczuk,tania,1999.0,12.26
2,5024,102,1,2026-01-28,telefon,Wrocław,1,Bieguni,obyczajowa,49.0,...,Olga,Tokarczuk,Polska,Nobel,49.0,1,Olga Tokarczuk,tania,1999.0,2.45
3,5038,102,2,2026-04-14,web,Kraków,1,Bieguni,obyczajowa,49.0,...,Olga,Tokarczuk,Polska,Nobel,98.0,4,Olga Tokarczuk,tania,1999.0,4.90
4,5042,102,1,2026-04-11,telefon,Warszawa,1,Bieguni,obyczajowa,49.0,...,Olga,Tokarczuk,Polska,Nobel,49.0,4,Olga Tokarczuk,tania,1999.0,2.45


## Ćwiczenie 4: pivot_table + crosstab — raport biznesowy

Krok 1 — pivot_table: kategoria × miesiąc

Zadanie 4a: Zbuduj tabelę przestawną z łącznym przychodem

In [30]:
pd.pivot_table(pelne, index='kategoria', columns='miesiac',
               values='wartosc', aggfunc='sum', fill_value=0)

miesiac,1,2,3,4
kategoria,,,,
fantasy,595.0,410.0,270.0,655
historyczna,1342.5,1790.9,634.4,369
obyczajowa,301.0,623.0,274.0,801
poezja,105.0,70.0,140.0,105
reportaz,416.0,0.0,728.0,0
sci-fi,765.0,615.0,672.0,282


Krok 2 — pivot_table z sumami brzegowymi

Zadanie 4b: Dodaj margins=True, margins_name='RAZEM'.

In [31]:
pd.pivot_table(pelne, index='kategoria', columns='miesiac',
               values='wartosc', aggfunc='sum', fill_value=0, margins=True, margins_name='RAZEM')

miesiac,1,2,3,4,RAZEM
kategoria,,,,,
fantasy,595.0,410.0,270.0,655,1930.0
historyczna,1342.5,1790.9,634.4,369,4136.8
obyczajowa,301.0,623.0,274.0,801,1999.0
poezja,105.0,70.0,140.0,105,420.0
reportaz,416.0,0.0,728.0,0,1144.0
sci-fi,765.0,615.0,672.0,282,2334.0
RAZEM,3524.5,3508.9,2718.4,2212,11963.8


Krok 3 — crosstab: rozkład zamówień per kanał i miasto

In [32]:
pd.crosstab(pelne['kanal'], pelne['miasto'])

miasto,Gdańsk,Kraków,Poznań,Warszawa,Wrocław
kanal,,,,,
aplikacja,7,1,5,6,4
telefon,1,4,1,3,2
web,5,9,14,9,9


Krok 4 — crosstab z normalize

Zadanie 4d: Przekształć w rozkład procentowy w wierszu:

In [36]:
pd.crosstab(pelne['kanal'], pelne['miasto'], normalize='index') * 100

miasto,Gdańsk,Kraków,Poznań,Warszawa,Wrocław
kanal,,,,,
aplikacja,30.434783,4.347826,21.739130,26.086957,17.391304
telefon,9.090909,36.363636,9.090909,27.272727,18.181818
web,10.869565,19.565217,30.434783,19.565217,19.565217


Kategoria historyczna wygenerowała największy przychód 4136.8 zł, najmniejszy zaś- poezja 420.0 zl.
Najmniejsze przychody przypadły na 4 miesiąc wysokości 2212 zł, gdzie  przychody z reportażu wyniosły 0 zł.
Najczęściej odwiedzanym kalałem jest web gdzie w Poznaniu ma 14 wartości co stanowi ok. 30.43% wszystkich wartości.